In [1]:
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy.ma as ma
import pandas as pd

# ---------------- USER INPUTS ----------------
THRESHOLD = 0.1  # meters; flooded if depth > THRESHOLD
boundary_zip = r"D:\Phd Research\GIS\Shape\land_Part_area_utm.zip"

ref_file     = r"D:\Phd Research\sim_depth_100yr.tif"

slr_minus    = r"D:\Phd Research\sim_depth_100yr_0.8SLR.tif"      # 20% less SLR
slr_plus     = r"D:\Phd Research\sim_depth_100yr_1.2SLR.tif"      # 20% more SLR

surge_minus  = r"D:\Phd Research\sim_depth_100yr_surge_M10.tif"   # 10% less surge height
surge_plus   = r"D:\Phd Research\sim_depth_100yr_surge_P10.tif"   # 10% more surge height

manning_minus = r"D:\Phd Research\sim_depth_100yr_surge_M20_Mannings.tif" # 20% less Manning's n
manning_plus  = r"D:\Phd Research\sim_depth_100yr_surge_P20_Mannings.tif" # 20% more Manning's n

flow_minus   = r"D:\Phd Research\sim_depth_100yr_surge_Q_M20.tif" # 20% less inland discharge
flow_plus    = r"D:\Phd Research\sim_depth_100yr_surge_Q_P20.tif" # 20% more inland discharge

# Fractional parameter perturbations ΔP/P0 used in your ± runs
DP_FRAC = {
    "SLR":        0.20,  # ±20%
    "Surge":      0.10,  # ±10%
    "Manning n":  0.20,  # ±20%
    "River Flow": 0.20,  # ±20%
}

# --------------- HELPERS ----------------
def read_boundary(zip_path: str) -> gpd.GeoDataFrame:
    """Read a zipped shapefile via geopandas."""
    try:
        return gpd.read_file(f"zip://{zip_path}")
    except Exception:
        return gpd.read_file(zip_path)

def flooded_extent_km2(raster_path: str, boundary_gdf: gpd.GeoDataFrame, threshold: float) -> float:
    """
    Flooded extent (km²) where depth > threshold, after clipping to boundary.
    Cleans NoData/NaN/Inf/negative depths.
    """
    with rasterio.open(raster_path) as src:
        if src.crs is None:
            raise ValueError(f"Raster CRS is None for: {raster_path}")
        bnd = boundary_gdf.to_crs(src.crs)
        out_img, out_transform = mask(src, bnd.geometry, crop=True)
        data = out_img[0].astype("float64")
        nodata = src.nodata

    # Clean
    if nodata is not None:
        data = np.where(data == nodata, np.nan, data)
    data = np.where((~np.isfinite(data)) | (data < 0), np.nan, data)

    # Flooded mask (> threshold)
    flooded = ma.array(data, mask=~(data > threshold))
    flooded_cells = flooded.count()

    # Cell area -> km²
    px_w, px_h = abs(out_transform.a), abs(out_transform.e)
    cell_area_m2 = px_w * px_h
    return (flooded_cells * cell_area_m2) / 1e6  # km²

def si_unitless(E0: float, E_minus: float, E_plus: float, dp_frac: float) -> float:
    """
    Sensitivity index (unitless): SI = (ΔE/E0) / (ΔP/P0),
    with central difference ΔE = (E+ - E-)/2.
    """
    if E0 == 0 or dp_frac == 0:
        return 0.0
    dE = (E_plus - E_minus) / 2.0
    return (dE / E0) / dp_frac

# --------------- MAIN ----------------
boundary = read_boundary(boundary_zip)

# Reference flooded area
E0 = flooded_extent_km2(ref_file, boundary, THRESHOLD)

# Drivers and file pairs
drivers = {
    "SLR":        (slr_minus,    slr_plus),
    "Surge":      (surge_minus,  surge_plus),
    "Manning n":  (manning_minus, manning_plus),
    "River Flow": (flow_minus,   flow_plus),
}

rows = []
for drv, (f_minus, f_plus) in drivers.items():
    Em = flooded_extent_km2(f_minus, boundary, THRESHOLD)
    Ep = flooded_extent_km2(f_plus,  boundary, THRESHOLD)
    dp = DP_FRAC[drv]

    # Unitless sensitivity
    SI = si_unitless(E0, Em, Ep, dp)

    # Per 1% change in driver:
    frac_change_per_1pct = SI * 0.01                 # ΔE/E0 per +1% in driver (fraction of E0)
    area_change_per_1pct = E0 * frac_change_per_1pct  # km² per +1% in driver

    rows.append({
        "Driver": drv,
        "E0_km2": E0,
        "E_minus_km2": Em,
        "E_plus_km2": Ep,
        "ΔP/P0_used": dp,
        "SI_unitless": SI,
        "Frac change per 1% driver (ΔE/E0)": frac_change_per_1pct,
        "Area change per 1% driver (km²)": area_change_per_1pct
    })

# Present results (sorted by absolute area change per 1%)
df = pd.DataFrame(rows)
df = df.sort_values(by="Area change per 1% driver (km²)", key=lambda s: np.abs(s), ascending=False)
pd.set_option("display.float_format", lambda v: f"{v:.6f}")
print("\n=== Sensitivity of Flooded Area per 1% Driver Change ===")
print(df.to_string(index=False))



=== Sensitivity of Flooded Area per 1% Driver Change ===
    Driver       E0_km2  E_minus_km2   E_plus_km2  ΔP/P0_used  SI_unitless  Frac change per 1% driver (ΔE/E0)  Area change per 1% driver (km²)
     Surge 13837.280000 13665.720000 13907.000000    0.100000     0.087185                           0.000872                        12.064000
       SLR 13837.280000 13798.320000 13876.600000    0.200000     0.014143                           0.000141                         1.957000
 Manning n 13837.280000 13829.880000 13857.840000    0.200000     0.005052                           0.000051                         0.699000
River Flow 13837.280000 13837.280000 13837.280000    0.200000     0.000000                           0.000000                         0.000000
